# Predicting next-month churn

Question: can we predict `churn_next_month` from a monthly account snapshot?

Design decisions (from checklist.md and columns.csv):
- **Leakage:** `lifetime_months` comes from a CRM export of 2026-01-05, i.e. after the snapshots; it encodes the cancellation outcome. Dropped. `account_ref` is an identifier; used only for grouping, never as a feature.
- **Test set:** the model is meant for future months, so the test set is the latest months. Accounts repeat across rows, so the test set also holds whole accounts: any account with a row in the test window is removed from the training rows entirely.
- **CV:** inside training, expanding-window forward splits by month, and accounts in the validation block are removed from that fold's training part.

In [ ]:
import numpy as np, pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import average_precision_score, roc_auc_score, f1_score, precision_score, recall_score
from sklearn.base import clone

df = pd.read_csv('data.csv')
dt = pd.to_datetime(df.snapshot_month)
df['month_idx'] = (dt.dt.year-2024)*12 + dt.dt.month-1
df['cal_month'] = dt.dt.month
print(df.shape, df.churn_next_month.mean())
print(df.groupby('snapshot_month').churn_next_month.mean().round(3).to_string())

Churn rate varies strongly by month (a large surge Nov 2024 - Jan 2025), so the data is non-stationary; a temporal test is the honest evaluation.

## 1. Test set (set aside before any modelling)
Test window: the last 3 snapshot months (2025-09 to 2025-11). Test accounts = accounts with a row in that window.

In [ ]:
test_start = df.month_idx.max() - 2
test_accounts = set(df.loc[df.month_idx >= test_start, 'account_ref'])
test = df[(df.month_idx >= test_start)]
train = df[(df.month_idx < test_start) & ~df.account_ref.isin(test_accounts)].reset_index(drop=True)
print(len(train), len(test), 'overlap accounts:', len(set(train.account_ref) & set(test.account_ref)))
print('train churn', train.churn_next_month.mean().round(3), 'test churn', test.churn_next_month.mean().round(3))

## 2. Candidates, named before any scoring
Features known at prediction time: plan, seats, monthly_fee, tenure_months, usage_hours, support_tickets (all recorded in the snapshot month). Feature set B adds calendar month (known in advance).

Models:
1. `baseline`: predicts the training prior (trivial)
2. `logreg_A`: logistic regression, features A
3. `logreg_B`: logistic regression, features B
4. `hgb_A`: gradient boosting, features A
5. `hgb_B`: gradient boosting, features B

Decision threshold: chosen on CV out-of-fold predictions by maximising F1. Selection metric: mean CV average precision (churn is imbalanced).

In [ ]:
NUM = ['seats','monthly_fee','tenure_months','usage_hours','support_tickets']
FA = ['plan']+NUM
FB = FA+['cal_month']
def pre(feats):
    num = [f for f in feats if f!='plan']
    return ColumnTransformer([('p', OneHotEncoder(handle_unknown='ignore'), ['plan']),
                              ('n', StandardScaler(), num)])
def make(name):
    kind, _, fs = name.partition('_')
    if kind=='baseline': return DummyClassifier(strategy='prior'), FA
    feats = FA if fs=='A' else FB
    est = LogisticRegression(max_iter=1000, C=1.0) if kind=='logreg' else HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, random_state=0)
    return Pipeline([('pre', pre(feats)), ('m', est)]), feats
names = ['baseline','logreg_A','logreg_B','hgb_A','hgb_B']

## 3. Cross-validation inside training rows (forward-chaining by month, whole accounts)

In [ ]:
def folds(train, n_folds=4, block=3):
    last = train.month_idx.max()
    for k in range(n_folds):
        v_end = last - block*(n_folds-1-k); v_start = v_end-block+1
        val = train[(train.month_idx>=v_start)&(train.month_idx<=v_end)]
        tr = train[(train.month_idx<v_start)&~train.account_ref.isin(set(val.account_ref))]
        yield tr, val

oof = {n: [] for n in names}; scores = {n: [] for n in names}
for tr, val in folds(train):
    for n in names:
        est, feats = make(n)
        est = clone(est).fit(tr[feats], tr.churn_next_month)   # preprocessing fitted within the fold (pipeline)
        p = est.predict_proba(val[feats])[:,1]
        oof[n].append((val.churn_next_month.values, p))
        scores[n].append(average_precision_score(val.churn_next_month, p))
    print(len(tr), len(val))
cv = pd.DataFrame(scores).T; cv['mean']=cv.mean(axis=1)
print(cv.round(3))

In [ ]:
# Threshold per model from pooled out-of-fold predictions
def best_thr(n):
    y = np.concatenate([a for a,_ in oof[n]]); p = np.concatenate([b for _,b in oof[n]])
    grid = np.linspace(0.02,0.8,79)
    f = [f1_score(y, p>=t, zero_division=0) for t in grid]
    return grid[int(np.argmax(f))], max(f)
for n in names: print(n, 'thr=%.2f cvF1=%.3f'%best_thr(n))

## 4. Choice (made on CV only, before the test set is touched)

In [ ]:
chosen = cv['mean'].idxmax()
thr, f = best_thr(chosen)
print('CHOSEN:', chosen, 'threshold', round(thr,2), 'CV AP', round(cv.loc[chosen,'mean'],3), 'CV F1', round(f,3))

The choice above is fixed: features, hyperparameters and threshold come from CV alone.

## 5. Refit on all training rows, score the test set once

In [ ]:
est, feats = make(chosen)
est = clone(est).fit(train[feats], train.churn_next_month)
p = est.predict_proba(test[feats])[:,1]; y = test.churn_next_month.values
pred = p>=thr
print('TEST  AP %.3f (prevalence %.3f)  AUC %.3f'%(average_precision_score(y,p), y.mean(), roc_auc_score(y,p)))
print('at CV threshold %.2f: precision %.3f recall %.3f F1 %.3f'%(thr, precision_score(y,pred,zero_division=0), recall_score(y,pred), f1_score(y,pred,zero_division=0)))

## Conclusion
The headline is the test-set average precision / AUC of the chosen model on the held-out latest months and unseen accounts, compared with the prevalence (= the AP of a no-skill model). Caveats: the churn rate is strongly seasonal and only ~2 years are available, so calendar-month effects are poorly identified; `lifetime_months` was excluded because it is not available at prediction time.